In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [11]:
B, T, C = 4, 8, 2
x = torch.randn(B, T, C) # Random numbers form a tensor of shape (B, T, C)
# With that out of the way, we can now add softmax to the matrix multiplication step, stepping closer to the self-attention block:
# New:
wei = torch.tril(torch.ones(T, T))       # Lower triangular matrix of ones
wei = wei / wei.sum(dim=1, keepdim=True) # Normalizing wei by dividing by the sum of each row
xbow2 = wei @ x                          # (T, T) @ (B, T, C) -> (B, T, T) @ (B, T, C) = (B, T, C)
print('Batch [0] xbow2:\n', xbow2[0], "\n")

# Newer:
tril = torch.tril(torch.ones(T, T))             # Lower triangular matrix of ones

wei = torch.zeros((T, T))                       # (T, T)
wei = wei.masked_fill(tril == 0, float('-inf')) # Masking all values in wei where tril == 0 with -inf
print('Wei_masked (T, T):\n', wei, "\n")
wei = F.softmax(wei, dim=-1)                    # (T, T)
print('Wei_softmaxed dim=-1 (T, T):\n', wei, "\n")

xbow3 = wei @ x                                 # (T, T) @ (B, T, C) -> (B, T, T) @ (B, T, C) = (B, T, C)
print('xbow3 Batch [0] wei @ x (T, T) @ (B, T, C) = (B, T, C):\n', xbow3[0], "\n")

torch.allclose(xbow2, xbow3)                     # True

Batch [0] xbow2:
 tensor([[-0.0399,  1.0038],
        [-0.5917,  1.4039],
        [-0.4041,  0.6649],
        [-0.0758,  0.2143],
        [ 0.0422,  0.0736],
        [ 0.0217,  0.2138],
        [-0.0597,  0.3414],
        [-0.0960,  0.3822]]) 

Wei_masked (T, T):
 tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]]) 

Wei_softmaxed dim=-1 (T, T):
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.200

True

In [45]:
# Version 4: Self-Attention
torch.manual_seed(1337)

B, T, C = 4, 8, 32        # Batch size, block size, vocab size (each token is a vector of size 32)
x = torch.randn(B, T, C)  # Random input of shape (B, T, C)

head_size = 16
key = nn.Linear(in_features=C, out_features=head_size, bias=False)   # No bias
query = nn.Linear(in_features=C, out_features=head_size, bias=False) # No bias
value = nn.Linear(in_features=C, out_features=head_size, bias=False) # No bias

k = key(x)   # (B, T, C) -> (B, T, head_size) | k = X @ W_key | k == x @ key.weight.T | (B, T, C) x (C, head_size) = (B, T, head_size)
q = query(x) # (B, T, C) -> (B, T, head_size) | q = X @ W_query |

wei = q @ k.transpose(-2, -1) * (head_size ** -0.5)  # | wei = q @ k.T | - (B, T, head_size) @ (B, head_size, T) = (B, T, T) (T is the block_size)
#print("wei = q @ k.T [0]:\n", wei[0])

tril = torch.tril(torch.ones(T, T))             # Lower triangular matrix of ones
#wei = torch.zeros((T, T))                      # (T, T)
wei = wei.masked_fill(tril == 0, float('-inf')) # Masking all values in wei where tril == 0 with -inf
wei = F.softmax(wei, dim=-1)                    # (T, T)
#out = wei @ x  # (T, T) @ (B, T, C) -> (B, T, T) @ (B, T, C) = (B, T, C)

v = value(x)   # | v = X @ W_value | (B, T, C) -> (B, T, head_size)
out = wei @ v  # | out = wei @ v | (B, T, T) @ (B, T, head_size) = (B, T, head_size)

print("wei = F.softmax(wei, dim=-1):[0]\n", wei[0])

wei = F.softmax(wei, dim=-1):[0]
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3966, 0.6034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3069, 0.2892, 0.4039, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3233, 0.2175, 0.2443, 0.2149, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1479, 0.2034, 0.1663, 0.1455, 0.3369, 0.0000, 0.0000, 0.0000],
        [0.1259, 0.2490, 0.1324, 0.1062, 0.3141, 0.0724, 0.0000, 0.0000],
        [0.1598, 0.1990, 0.1140, 0.1125, 0.1418, 0.1669, 0.1061, 0.0000],
        [0.0845, 0.1197, 0.1078, 0.1537, 0.1086, 0.1146, 0.1558, 0.1553]],
       grad_fn=<SelectBackward0>)


In [40]:
wei

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
         [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
         [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
         [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],

        [[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1687, 0.8313, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2477, 0.0514, 0.7008, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4410, 0.0957, 0.3747, 0.0887, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0069, 0.0456, 0.0300, 0.7748, 0.1427, 0.0000, 0.0000, 0.0000],
         [0.0660, 0.089